# Train Random Forest on Traditional Opcode N-gram Frequencies

Trains a RandomForestClassifier on sparse frequency vectors built from
consecutive opcode mnemonics within each function. N-gram windows reset at
function boundaries; no CFG or DFG paths are used. The final cell also
reports in-sample training-set performance; these results can be
optimistic because the same samples were used to fit the model.

Run Pass1BuildVocabulary.py and pass2BuildDataset.py before training.


In [9]:
# ---------------------------------------------------------------------------
# Cell 1: Setup, imports, configuration
# ---------------------------------------------------------------------------
import os
import gc
import glob
import json
import time
import joblib
import numpy as np
import scipy.sparse as sp
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV

# ---- Configuration ----
NGRAM_COUNT = 4  # n-gram size used for feature extraction; change to match dataset
TRAIN_DIR = os.path.join("feature_datasets", f"{NGRAM_COUNT}gram", "train")  # directory holding X_batch_iii.npz / y_batch_iii.npy
OUTPUT_DIR = os.path.join("outputs", f"{NGRAM_COUNT}gram")
# Auto-detect chunk count from the directory instead of hardcoding it --
# train/ and test/ are not guaranteed to have the same number of chunks.
N_CHUNKS = len(glob.glob(os.path.join(TRAIN_DIR, "X_batch_*.npz")))
if N_CHUNKS == 0:
    raise FileNotFoundError(f"No X_batch_*.npz files found in {TRAIN_DIR}")

# Create the nested output directory if it doesn't already exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"NGRAM_COUNT = {NGRAM_COUNT}")
print(f"TRAIN_DIR   = {TRAIN_DIR}")
print(f"OUTPUT_DIR  = {OUTPUT_DIR}")
print(f"N_CHUNKS    = {N_CHUNKS} (auto-detected)")


NGRAM_COUNT = 4
TRAIN_DIR   = feature_datasets\4gram\train
OUTPUT_DIR  = outputs\4gram
N_CHUNKS    = 36 (auto-detected)


In [10]:
# ---------------------------------------------------------------------------
# Cell 2: Memory-efficient chunk loader
# ---------------------------------------------------------------------------
def load_chunks(data_dir, n_chunks, prefix_features="X_batch", prefix_labels="y_batch"):
    """
    Memory-efficient loader for sparse feature chunks + label chunks.

    Reads n_chunks pairs of files:
        {data_dir}/{prefix_features}_{i}.npz  -> scipy.sparse CSR matrix
        {data_dir}/{prefix_labels}_{i}.npy    -> numpy label array (1D)

    Stacks all chunks into a single CSR matrix (via scipy.sparse.vstack)
    and a single label array (via np.concatenate). Periodically triggers
    gc.collect() to release intermediate memory as chunks accumulate,
    which matters when working with ~313k-wide sparse matrices on a
    memory-constrained VM.

    Parameters
    ----------
    data_dir : str
        Directory containing the chunk files.
    n_chunks : int
        Number of chunk files to load (indexed 0..n_chunks-1).
    prefix_features : str
        Filename prefix for feature chunks (default "X_chunk").
    prefix_labels : str
        Filename prefix for label chunks (default "y_chunk").

    Returns
    -------
    X : scipy.sparse.csr_matrix, shape (n_samples, n_features)
    y : np.ndarray, shape (n_samples,)
    """
    X_parts = []
    y_parts = []

    for i in range(n_chunks):
        x_path = os.path.join(data_dir, f"{prefix_features}_{i:03d}.npz")
        y_path = os.path.join(data_dir, f"{prefix_labels}_{i:03d}.npy")

        if not os.path.exists(x_path):
            raise FileNotFoundError(f"Missing feature chunk: {x_path}")
        if not os.path.exists(y_path):
            raise FileNotFoundError(f"Missing label chunk: {y_path}")

        # Ensure CSR format explicitly (npz may store CSR/CSC/COO)
        X_chunk = sp.load_npz(x_path).tocsr()
        y_chunk = np.load(y_path)

        if X_chunk.shape[0] != y_chunk.shape[0]:
            raise ValueError(
                f"Row mismatch in chunk {i}: "
                f"X has {X_chunk.shape[0]} rows, y has {y_chunk.shape[0]}"
            )

        X_parts.append(X_chunk)
        y_parts.append(y_chunk)

        print(f"[{i + 1}/{n_chunks}] loaded {x_path}  shape={X_chunk.shape}")

        # Periodic GC every 6 chunks to release freed intermediate memory
        if (i + 1) % 6 == 0:
            gc.collect()

    print("Stacking chunks with scipy.sparse.vstack / np.concatenate ...")
    X = sp.vstack(X_parts, format="csr")
    y = np.concatenate(y_parts)

    # Drop references to per-chunk lists and force a final collection
    del X_parts, y_parts
    gc.collect()

    density = X.nnz / (X.shape[0] * X.shape[1])
    print(f"Final stacked matrix: X.shape={X.shape}, y.shape={y.shape}, "
          f"X.nnz={X.nnz:,} ({density:.6%} density)")

    return X, y


In [11]:
# ---------------------------------------------------------------------------
# Cell 3: RandomForestClassifier + GridSearchCV configuration
# ---------------------------------------------------------------------------
# Settings kept FIXED (not searched) -- these control memory footprint
# against ~313,000 sparse features and are not really "model quality"
# hyperparameters:
#   - max_features="sqrt": each split only evaluates sqrt(n_features)
#     candidates instead of all 313k, bounding per-split cost/memory.
#   - n_jobs=4: parallelism cap for a single RandomForest fit. NOTE this is
#     deliberately NOT also parallelized at the GridSearchCV level (see
#     n_jobs=1 below) -- nesting two parallel layers would multiply worker
#     processes together and risk a severe memory spike.
BASE_RF_PARAMS = dict(
    max_features="sqrt",
    n_jobs=4,
    random_state=42,
)

# Hyperparameters GridSearchCV will search over. Keep this modest:
# every combination is fit CV_FOLDS times, so grid size x folds is your
# real training cost multiplier. Trim this grid (fewer values, fewer
# folds) if a full sweep is too slow for your dataset size / hardware.
PARAM_GRID = {
    "n_estimators": [200],
    "max_depth": [15, 20, 30, 40],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
}

CV_FOLDS = 3              # k-fold cross-validation within the training set
SCORING = "f1_weighted"   # weighted by class support -- robust to class imbalance

n_combinations = 1
for _values in PARAM_GRID.values():
    n_combinations *= len(_values)
total_fits = n_combinations * CV_FOLDS
print(f"Grid size: {n_combinations} combinations x {CV_FOLDS} folds = {total_fits} total fits")

base_rf = RandomForestClassifier(**BASE_RF_PARAMS)

grid_search = GridSearchCV(
    estimator=base_rf,
    param_grid=PARAM_GRID,
    cv=CV_FOLDS,
    scoring=SCORING,
    n_jobs=1,        # outer loop kept serial -- RF itself already parallelizes (n_jobs=4 above)
    verbose=2,
    refit=True,      # after search, automatically refits the best params on the FULL training set
)
print(grid_search)


Grid size: 16 combinations x 3 folds = 48 total fits
GridSearchCV(cv=3, estimator=RandomForestClassifier(n_jobs=4, random_state=42),
             n_jobs=1,
             param_grid={'max_depth': [15, 20, 30, 40],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5], 'n_estimators': [200]},
             scoring='f1_weighted', verbose=2)


In [12]:
# ---------------------------------------------------------------------------
# Cell 4: Run GridSearchCV, export the best model + grid search results
# ---------------------------------------------------------------------------
# Load & stack all N_CHUNKS training chunks
t0 = time.time()
X_train, y_train = load_chunks(TRAIN_DIR, N_CHUNKS)
print(f"Loaded training data in {time.time() - t0:.1f}s")

# Run the grid search directly on the sparse CSR matrix -- NEVER call
# .toarray(). GridSearchCV internally handles the CV splitting and, with
# refit=True, automatically refits the best-scoring params on the FULL
# training set afterward -- no separate manual .fit() call needed.
print(f"Running GridSearchCV ({total_fits} total fits) ...")
t0 = time.time()
grid_search.fit(X_train, y_train)
print(f"GridSearchCV completed in {time.time() - t0:.1f}s")

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV {SCORING} score: {grid_search.best_score_:.4f}")

# The refit-on-full-data estimator is what we actually want to ship
rf = grid_search.best_estimator_

# Report in-sample performance on the same training data used to fit rf.
positive_class_indices = np.flatnonzero(rf.classes_ == 1)
if positive_class_indices.size != 1:
    raise ValueError(f"Expected class label 1, found classes {rf.classes_}")
positive_class_index = int(positive_class_indices[0])
y_train_pred = rf.predict(X_train)
y_train_proba = rf.predict_proba(X_train)[:, positive_class_index]
training_report = classification_report(y_train, y_train_pred, output_dict=True)
training_accuracy = accuracy_score(y_train, y_train_pred)
train_tn, train_fp, train_fn, train_tp = confusion_matrix(
    y_train, y_train_pred, labels=[0, 1]
).ravel()
train_negative_count = train_tn + train_fp
if train_negative_count == 0:
    raise ValueError("Training false positive rate is undefined without class-0 samples")
training_metrics = {
    "evaluation_dataset": "training",
    "evaluation_type": "in_sample",
    "evaluation_note": "In-sample scores can be optimistic because these samples were used to fit the model.",
    "prediction_rule": "RandomForestClassifier.predict default class decision",
    "n_samples": int(X_train.shape[0]),
    "n_features": int(X_train.shape[1]),
    "accuracy": float(training_accuracy),
    "false_positive_rate": float(train_fp / train_negative_count),
    "true_negatives": int(train_tn),
    "false_positives": int(train_fp),
    "false_negatives": int(train_fn),
    "true_positives": int(train_tp),
    "roc_auc": float(roc_auc_score(y_train, y_train_proba)),
    "pr_auc": float(average_precision_score(y_train, y_train_proba)),
    "classification_report": training_report,
}
training_metrics_path = os.path.join(OUTPUT_DIR, "training_metrics.json")
with open(training_metrics_path, "w") as f:
    json.dump(training_metrics, f, indent=2)
print("In-sample training-set performance (same samples used to fit the model):")
print(f"  Samples: {training_metrics['n_samples']}")
print(f"  Accuracy: {training_accuracy:.4%}")
print(f"  Weighted F1: {training_report['weighted avg']['f1-score']:.4%}")
print(f"Training metrics saved to: {training_metrics_path}")

# Free training data before persisting anything
del X_train, y_train
gc.collect()

# --- Persist the trained (best) model ---
model_path = os.path.join(OUTPUT_DIR, "rf_model.joblib")
joblib.dump(rf, model_path, compress=3)
print(f"Model saved to: {model_path}")

# --- Persist the final hyperparameters to a separate JSON file ---
# best_params_ only contains the SEARCHED params -- merge in the fixed
# ones too, so the file reflects the complete param set the final model
# was actually fit with, not just the subset GridSearchCV swept.
final_params = {**BASE_RF_PARAMS, **grid_search.best_params_}

grid_search_output = {
    "best_params": final_params,
    "best_cv_score": grid_search.best_score_,
    "scoring": SCORING,
}

grid_search_path = os.path.join(OUTPUT_DIR, "grid_search_results.json")
with open(grid_search_path, "w") as f:
    json.dump(grid_search_output, f, indent=2)
print(f"Final hyperparameters saved to: {grid_search_path}")


[1/36] loaded feature_datasets\4gram\train\X_batch_000.npz  shape=(100, 3446611)
[2/36] loaded feature_datasets\4gram\train\X_batch_001.npz  shape=(100, 3446611)
[3/36] loaded feature_datasets\4gram\train\X_batch_002.npz  shape=(100, 3446611)
[4/36] loaded feature_datasets\4gram\train\X_batch_003.npz  shape=(100, 3446611)
[5/36] loaded feature_datasets\4gram\train\X_batch_004.npz  shape=(100, 3446611)
[6/36] loaded feature_datasets\4gram\train\X_batch_005.npz  shape=(100, 3446611)
[7/36] loaded feature_datasets\4gram\train\X_batch_006.npz  shape=(100, 3446611)
[8/36] loaded feature_datasets\4gram\train\X_batch_007.npz  shape=(100, 3446611)
[9/36] loaded feature_datasets\4gram\train\X_batch_008.npz  shape=(100, 3446611)
[10/36] loaded feature_datasets\4gram\train\X_batch_009.npz  shape=(100, 3446611)
[11/36] loaded feature_datasets\4gram\train\X_batch_010.npz  shape=(100, 3446611)
[12/36] loaded feature_datasets\4gram\train\X_batch_011.npz  shape=(100, 3446611)
[13/36] loaded feature_da